In [43]:
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path

1. Defining configuration

In [44]:
IMAGE_SIZE = 112
PATCH_SIZE = 8
HIDDEN_SIZE = 128
NUM_HEADS = 4
MLP_SIZE = 256
SEED = 42
NUM_TRANSFORMER_BLOCKS = 4
EMBEDDING_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 20
AUTOTUNE = tf.data.AUTOTUNE

2. Loading dataset

In [45]:
df_train = pd.read_csv('./dataset/Dataset.csv', dtype = 'str')

In [46]:
print("Dataset shape:", df_train.shape)
print(df_train.head())

Dataset shape: (2562, 2)
                        id             label
0  Robert Downey Jr_87.jpg  Robert Downey Jr
1       Lisa Kudrow_64.jpg       Lisa Kudrow
2   Ellen Degeneres_34.jpg   Ellen Degeneres
3      Billie Eilish_3.jpg     Billie Eilish
4    Hrithik Roshan_35.jpg    Hrithik Roshan


In [47]:
df_train = df_train.dropna(subset=["id", "label"])

In [ ]:
ROOT = Path.cwd()
DATASET = ROOT / "dataset" / "Faces"
DATASET_1 = ROOT / "dataset" / "Original Images"

def load_images(folder):
    images = []

    for path in folder.rglob("*"):
        if path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            image = cv2.imread(str(path))

            if image is not None:
                images.append({
                    "path": path,
                    "image": image
                })

    return images

faces = load_images(DATASET)

print(f"Loaded {len(faces)} images")

Loaded 2562 images


In [49]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_1,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_1,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_ds.class_names
num_identities = len(class_names)

print("Number of identities:", num_identities)
print("Identities:", class_names)

# Convert pixel values from [0, 255] to [0, 1].
def normalize_images(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

train_ds = (
    train_ds
    .map(normalize_images, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(normalize_images, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

Found 2562 files belonging to 31 classes.
Using 2050 files for training.
Found 2562 files belonging to 31 classes.
Using 512 files for validation.
Number of identities: 31
Identities: ['Akshay Kumar', 'Alexandra Daddario', 'Alia Bhatt', 'Amitabh Bachchan', 'Andy Samberg', 'Anushka Sharma', 'Billie Eilish', 'Brad Pitt', 'Camila Cabello', 'Charlize Theron', 'Claire Holt', 'Courtney Cox', 'Dwayne Johnson', 'Elizabeth Olsen', 'Ellen Degeneres', 'Henry Cavill', 'Hrithik Roshan', 'Hugh Jackman', 'Jessica Alba', 'Kashyap', 'Lisa Kudrow', 'Margot Robbie', 'Marmik', 'Natalie Portman', 'Priyanka Chopra', 'Robert Downey Jr', 'Roger Federer', 'Tom Cruise', 'Vijay Deverakonda', 'Virat Kohli', 'Zac Efron']


In [65]:
for image_path in faces:
    image = cv2.imread(str(image_path))

    if image is None:
        continue

    print(image_path.name, image.shape)

3. Preprocessing data

In [51]:
class PatchEncoder(tf.keras.layers.Layer):

    def __init__(self, image_size, patch_size, hidden_size):
        super().__init__()

        self.patch_size = patch_size
        self.hidden_size = hidden_size

        self.num_patches = (image_size // patch_size) ** 2

        self.patch_dim = patch_size * patch_size * 3

        self.projection = tf.keras.layers.Dense(hidden_size)

        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=self.num_patches,
            output_dim=hidden_size
        )

    def call(self, images):

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        batch_size = tf.shape(images)[0]

        patches = tf.reshape(
            patches,
            [batch_size, self.num_patches, self.patch_dim]
        )

        patches = tf.ensure_shape(
            patches,
            [None, self.num_patches, self.patch_dim]
        )

        positions = tf.range(self.num_patches)

        encoded = self.projection(patches)

        encoded += self.position_embedding(positions)

        return encoded

In [52]:

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, hidden_size, num_heads, mlp_size, dropout=0.1):
        super().__init__()

        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.attention = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=hidden_size // num_heads,
            dropout=dropout
        )

        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(mlp_size, activation="gelu"),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(hidden_size),
            tf.keras.layers.Dropout(dropout)
        ])

    def call(self, x, training=None):
        normalized = self.norm1(x)

        attention_output = self.attention(
            normalized,
            normalized,
            training=training
        )

        x = x + self.dropout1(
            attention_output,
            training=training
        )

        x = x + self.mlp(
            self.norm2(x),
            training=training
        )

        return x


4. Building model

In [53]:

def build_vit_face_model(num_identities):
    inputs = tf.keras.layers.Input(
        shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        name="face_image"
    )

    x = PatchEncoder(
        IMAGE_SIZE,
        PATCH_SIZE,
        HIDDEN_SIZE
    )(inputs)

    for _ in range(NUM_TRANSFORMER_BLOCKS):
        x = TransformerBlock(
            HIDDEN_SIZE,
            NUM_HEADS,
            MLP_SIZE
        )(x)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(256, activation="gelu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    embedding = tf.keras.layers.Dense(
        EMBEDDING_SIZE,
        name="raw_embedding"
    )(x)

    embedding = tf.keras.layers.UnitNormalization(
        axis=-1,
        name="embedding"
    )(embedding)

    identity = tf.keras.layers.Dense(
        num_identities,
        activation="softmax",
        name="identity"
    )(embedding)

    training_model = tf.keras.Model(
        inputs=inputs,
        outputs=identity,
        name="vit_face_classifier"
    )

    embedding_model = tf.keras.Model(
        inputs=inputs,
        outputs=embedding,
        name="vit_face_embedding"
    )

    return training_model, embedding_model


5. Training model

In [ ]:
training_model, embedding_model = build_vit_face_model(
    num_identities=num_identities
)

training_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )
]

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

# embedding_model.save("vit_face_embedding.keras")


Epoch 1/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.0411 - loss: 3.4204 - val_accuracy: 0.0410 - val_loss: 3.4202
Epoch 2/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - accuracy: 0.0519 - loss: 3.4001 - val_accuracy: 0.0469 - val_loss: 3.4120
Epoch 3/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.0508 - loss: 3.3934 - val_accuracy: 0.0527 - val_loss: 3.4011
Epoch 4/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.0626 - loss: 3.3841 - val_accuracy: 0.0508 - val_loss: 3.3931
Epoch 5/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 179s 2s/step - accuracy: 0.0656 - loss: 3.3705 - val_accuracy: 0.0527 - val_loss: 3.3794


6. Comparing faces

In [ ]:
def load_face(path):
    image = tf.keras.utils.load_img(
        path,
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        color_mode="rgb"
    )

    image = tf.keras.utils.img_to_array(image)
    image = image / 255.0

    return tf.expand_dims(image, axis=0)


def get_face_embedding(image_path):
    image = load_face(image_path)

    embedding = embedding_model(
        image,
        training=False
    )

    return embedding.numpy()[0]


def cosine_similarity(embedding1, embedding2):
    embedding1 = embedding1 / (
        np.linalg.norm(embedding1) + 1e-10
    )
    embedding2 = embedding2 / (
        np.linalg.norm(embedding2) + 1e-10
    )

    return float(np.dot(embedding1, embedding2))


face1 = get_face_embedding(DATASET/"Akshay Kumar_0.jpg")
face2 = get_face_embedding(DATASET/"Henry Cavill_5.jpg")

similarity = cosine_similarity(face1, face2)

print("Cosine similarity:", similarity)


Cosine similarity: 0.9896178245544434


In [59]:
threshold = 0.70  # Example only; NOT a validated threshold.

if similarity >= threshold:
    print("Similar embeddings")
else:
    print("Dissimilar embeddings")

Similar embeddings
